# 2.5 Kavram Ağı — Concept Term ARM Upload

**Plan:** `Page_Design/Sayfa_Plani_v2/2.5_kavram_agi.rtf` — Drive parquet → Supabase taşıma (Omer onayı 2026-05-10, A yolu).

**Patern:** COPY FROM STDIN + post-load CREATE INDEX (Compute Small'da bile 5-10× hız kazancı vs `execute_values` + migration index).

**Veri kaynağı (ENVANTER.md kanıt A):**

| Tablo | Drive parquet | Satır | Core kolonlar |
|---|---|---|---|
| fact_term_arm_static    | `~/Dataleak/facts/fact_term_arm_static.parquet`   | 297,855 | term_a, term_b, lift, npmi, support, conviction |
| fact_term_arm_temporal  | `~/Dataleak/facts/fact_term_arm_temporal.parquet` | 547,824 | term_a, term_b, year, delta_lift |
| dim_term_community      | `~/Dataleak/facts/dim_term_community.parquet`     |   4,516 | term, community_id |

**Önkoşul (notebook DIŞINDA):**
```bash
psql "$SUPABASE_DB_URL" -f db/migrations/0021_concept_term_arm.sql
```
K-029 dersi: Supabase Dashboard SQL Editor atomic rollback **silent fail**. psql/asyncpg zorunlu. DB_URL Session Pooler `:5432` (transaction `:6543` ISP blok).

**Done-of-Definition:**
- 3 tablo Supabase'de var, satır sayısı parquet ±0
- 5 secondary index oluşturuldu (post-load)
- ANALYZE her tabloya çalıştırıldı
- Sample query: anchor 1-hop top-NPMI 5 satır döner

**Checkpoint + log (Drive):**
```
~/Dataleak/papermind_app_uploads/concept_terms/
├── state_<table>.json          (table, parquet_rows, rows_done, started_at, completed_at)
└── log_concept_terms.jsonl     (her batch sonrası 1 satır: ts, table, rows_done, rate, eta, pct)
```
Restart-safe: state.rows_done ≥ parquet_rows ise tablo SKIP. Partial resume row-skip ile (pyarrow row-group deterministic ordering).

## Cell 1 — Setup (Drive mount + install + DB_URL + parquet path verify)

In [ ]:
!pip -q install psycopg2-binary==2.9.9 pyarrow==17.0.0 polars==1.8.2 pandas==2.2.2

import os, io, csv, json, time
from datetime import datetime, timedelta
from pathlib import Path

import psycopg2
import pyarrow.parquet as pq
import polars as pl
import pandas as pd

from google.colab import drive, userdata
drive.mount('/content/drive')

# DB_URL Colab Secrets'tan (K-029: Session Pooler :5432 zorunlu)
DB_URL = userdata.get('SUPABASE_DB_URL')
assert DB_URL, 'SUPABASE_DB_URL Colab Secrets\'ta tanımlı olmalı (Settings → Secrets).'
assert ':5432/' in DB_URL, 'K-029: Session Pooler :5432 kullan (transaction :6543 ISP blok).'

# Drive path'leri
DRIVE_FACTS = Path('/content/drive/MyDrive/Dataleak/facts')
PARQUETS = {
    'fact_term_arm_static':   DRIVE_FACTS / 'fact_term_arm_static.parquet',
    'fact_term_arm_temporal': DRIVE_FACTS / 'fact_term_arm_temporal.parquet',
    'dim_term_community':     DRIVE_FACTS / 'dim_term_community.parquet',
}

WORK_DIR = Path('/content/drive/MyDrive/Dataleak/papermind_app_uploads/concept_terms')
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = WORK_DIR / 'log_concept_terms.jsonl'

print(f'DB host: {DB_URL.split("@")[1].split("/")[0] if "@" in DB_URL else "masked"}')
print(f'Work dir: {WORK_DIR}')
print(f'Log: {LOG_PATH}\n')
print('Parquet kanıt:')
for table, path in PARQUETS.items():
    assert path.exists(), f'YOK: {path}'
    n = pq.ParquetFile(path).metadata.num_rows
    print(f'  ✓ {table}: {path.name} ({n:,} satır)')

## Cell 2 — Parquet schema doğrulama (L-022)

ENVANTER plan-time şema güvenilir değil; her parquet'in gerçek kolonlarını oku, core kolonları doğrula, fazlasını `extra` JSONB'ye haritala.

In [ ]:
CORE_COLS = {
    'fact_term_arm_static':   ['term_a', 'term_b', 'lift', 'npmi', 'support'],
    'fact_term_arm_temporal': ['term_a', 'term_b', 'year', 'delta_lift'],
    'dim_term_community':     ['term', 'community_id'],
}

EXTRA_COLS = {}
print('=== Parquet schema (L-022 doğrulama) ===\n')
for table, path in PARQUETS.items():
    schema = pl.scan_parquet(str(path)).collect_schema()
    cols = list(schema.names())
    core = CORE_COLS[table]
    missing = [c for c in core if c not in cols]
    if missing:
        raise SystemExit(f'  ✗ {table}: core kolon eksik {missing} | parquet kolonları: {cols}')
    extra = [c for c in cols if c not in core]
    EXTRA_COLS[table] = extra
    print(f'  {table}:')
    print(f'    parquet cols ({len(cols)}): {cols}')
    print(f'    core ✓:    {core}')
    print(f'    → extra:   {extra if extra else "(yok)"}\n')

## Cell 3 — Önkoşul: migration 0021 apply'lı, 3 tablo var

Bu hücre sadece doğrular; migration uygulamaz. Migration eksikse SystemExit + lokalden apply komutu basar.

In [ ]:
EXPECTED_TABLES = list(PARQUETS.keys())

print('=== Önkoşul: 0021 apply + 3 tablo varlık ===')
with psycopg2.connect(DB_URL) as conn, conn.cursor() as cur:
    cur.execute("SELECT version FROM public.schema_migrations WHERE version = '0021_concept_term_arm'")
    if not cur.fetchone():
        print('  ✗ 0021_concept_term_arm migration uygulanmamış.')
        print('  Lokalden uygula:')
        print('    psql "$SUPABASE_DB_URL" -f db/migrations/0021_concept_term_arm.sql')
        raise SystemExit('Migration eksik — apply edip Cell 3\'ü tekrar koş.')
    print('  ✓ schema_migrations: 0021_concept_term_arm')

    for t in EXPECTED_TABLES:
        cur.execute('SELECT to_regclass(%s)', (f'public.{t}',))
        if cur.fetchone()[0] is None:
            raise SystemExit(f'  ✗ Tablo yok: public.{t}')
        cur.execute(f'SELECT COUNT(*) FROM public.{t}')
        n = cur.fetchone()[0]
        parquet_n = pq.ParquetFile(PARQUETS[t]).metadata.num_rows
        status = '(boş, yüklenecek)' if n == 0 else f'(zaten {n:,} satır — resume veya skip)'
        print(f'  ✓ public.{t}: db={n:,} / parquet={parquet_n:,} {status}')

## Cell 4 — `copy_upload` helper (COPY FROM STDIN + checkpoint + JSONL log)

**Restart-safe:** state.json'da rows_done ≥ parquet_rows ise tablo skip. Partial resume row-skip ile (pyarrow row-group deterministic ordering).  
**NULL marker:** CSV içinde `\N` literal; Postgres COPY `NULL '\N'` ile yorumlar.  
**Extra JSONB:** parquet'in core olmayan kolonları dict → JSON string → CSV son alan.

In [ ]:
def _norm_cell(v):
    """numpy/pandas NA → None; numpy scalar → python primitive."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(v, 'item'):
        return v.item()
    return v

def _serialize_extra(row, extra_cols):
    d = {c: _norm_cell(row[c]) for c in extra_cols}
    return json.dumps(d, default=str, ensure_ascii=False)

def copy_upload(table, parquet_path, core_cols, extra_cols,
                batch_rows=20000, db_url=DB_URL,
                work_dir=WORK_DIR, log_path=LOG_PATH):
    """COPY FROM STDIN bulk loader. Per-batch commit + Drive checkpoint + JSONL log."""
    state_path = work_dir / f'state_{table}.json'
    pf = pq.ParquetFile(parquet_path)
    total = pf.metadata.num_rows

    # Resume state
    if state_path.exists():
        state = json.loads(state_path.read_text())
        if state.get('rows_done', 0) >= total:
            print(f'  [SKIP-COMPLETE] {table}: {state["rows_done"]:,} rows (state.completed_at={state.get("completed_at", "?")})')
            return state
    else:
        state = {'table': table, 'parquet_rows': total, 'rows_done': 0,
                 'started_at': datetime.utcnow().isoformat() + 'Z'}

    skip_until = state.get('rows_done', 0)
    if skip_until > 0:
        print(f'  [RESUME] {table}: skip first {skip_until:,} rows')

    cols_quoted = ', '.join(f'"{c}"' for c in core_cols)
    has_extra = bool(extra_cols)
    if has_extra:
        cols_quoted += ', "extra"'
    copy_sql = f"COPY public.{table} ({cols_quoted}) FROM STDIN WITH (FORMAT csv, NULL '\\N')"

    rows_done_start = state.get('rows_done', 0)
    rows_done = rows_done_start
    seen = 0
    t0 = time.time()
    last_ckpt = t0
    last_print = t0

    with psycopg2.connect(db_url) as conn:
        with conn.cursor() as cur:
            for batch in pf.iter_batches(batch_size=batch_rows):
                bsize = batch.num_rows
                # Resume: tüm batch zaten yüklenmiş mi
                if seen + bsize <= skip_until:
                    seen += bsize
                    continue

                df = batch.to_pandas()
                start_idx = max(0, skip_until - seen) if seen < skip_until else 0

                buf = io.StringIO()
                w = csv.writer(buf, quoting=csv.QUOTE_MINIMAL)
                for i in range(start_idx, len(df)):
                    row = df.iloc[i]
                    csv_row = []
                    for c in core_cols:
                        v = _norm_cell(row[c])
                        csv_row.append(r'\N' if v is None else v)
                    if has_extra:
                        csv_row.append(_serialize_extra(row, extra_cols))
                    w.writerow(csv_row)

                buf.seek(0)
                cur.copy_expert(copy_sql, buf)
                conn.commit()

                actual = bsize - start_idx
                rows_done += actual
                seen += bsize

                # JSONL log
                now = time.time()
                elapsed = now - t0
                rate = (rows_done - rows_done_start) / max(0.001, elapsed)
                eta = (total - rows_done) / max(0.001, rate) if rate > 0 else 0
                pct = round(100 * rows_done / total, 2)
                event = {
                    'ts': datetime.utcnow().isoformat() + 'Z',
                    'table': table, 'rows_done': rows_done, 'parquet_rows': total,
                    'rate_per_s': round(rate, 1), 'eta_s': int(eta), 'pct': pct,
                }
                with open(log_path, 'a') as f:
                    f.write(json.dumps(event) + '\n')

                # Drive checkpoint per 30s (sync thrash önle)
                if now - last_ckpt > 30:
                    state['rows_done'] = rows_done
                    state['last_at'] = event['ts']
                    state_path.write_text(json.dumps(state, indent=2))
                    last_ckpt = now

                # Console print per 5s
                if now - last_print > 5:
                    print(f'  {rows_done:,}/{total:,} ({pct:.1f}%)  '
                          f'rate={rate:.0f}/s  ETA={timedelta(seconds=int(eta))}', flush=True)
                    last_print = now

    state['rows_done'] = rows_done
    state['completed_at'] = datetime.utcnow().isoformat() + 'Z'
    state['duration_s'] = round(time.time() - t0, 1)
    state_path.write_text(json.dumps(state, indent=2))
    print(f'  ✅ {table}: {rows_done:,}/{total:,} rows | {state["duration_s"]:.1f}s')
    return state

print('copy_upload helper hazır.')

## Cell 5 — Upload (3 tablo, küçükten büyüğe smoke-first)

Sıra: dim_term_community (4.5K — smoke) → fact_term_arm_static (297K) → fact_term_arm_temporal (547K).  
Toplam tahmini 3-7 dk (Compute Small + COPY).

In [ ]:
print('=== Upload 1/3: dim_term_community (smoke) ===')
copy_upload(
    table='dim_term_community',
    parquet_path=PARQUETS['dim_term_community'],
    core_cols=CORE_COLS['dim_term_community'],
    extra_cols=EXTRA_COLS['dim_term_community'],
    batch_rows=5000,
)

print('\n=== Upload 2/3: fact_term_arm_static ===')
copy_upload(
    table='fact_term_arm_static',
    parquet_path=PARQUETS['fact_term_arm_static'],
    core_cols=CORE_COLS['fact_term_arm_static'],
    extra_cols=EXTRA_COLS['fact_term_arm_static'],
    batch_rows=20000,
)

print('\n=== Upload 3/3: fact_term_arm_temporal ===')
copy_upload(
    table='fact_term_arm_temporal',
    parquet_path=PARQUETS['fact_term_arm_temporal'],
    core_cols=CORE_COLS['fact_term_arm_temporal'],
    extra_cols=EXTRA_COLS['fact_term_arm_temporal'],
    batch_rows=20000,
)

print('\n✅ 3 upload tamam.')

## Cell 6 — Post-load: CREATE INDEX (5) + ANALYZE + verify + sample query

**Index seçimi:**
- `fact_term_arm_static`: anchor 1-hop alt-graf için 2 index (term_a, npmi DESC) + (term_b, npmi DESC) — sorgu `WHERE term_a = ? OR term_b = ?` her iki yönü destekler.
- `fact_term_arm_temporal`: trend okuma için (term_a, year) + (term_b, year).
- `dim_term_community`: community-batch lookup için (community_id).

Boş tablo değil → CREATE INDEX serial DDL (CONCURRENTLY gereksiz; tek seferlik bulk-load post).

In [ ]:
INDEXES = [
    ('idx_arm_static_term_a_npmi',
     'CREATE INDEX IF NOT EXISTS idx_arm_static_term_a_npmi '
     'ON public.fact_term_arm_static (term_a, npmi DESC NULLS LAST);'),
    ('idx_arm_static_term_b_npmi',
     'CREATE INDEX IF NOT EXISTS idx_arm_static_term_b_npmi '
     'ON public.fact_term_arm_static (term_b, npmi DESC NULLS LAST);'),
    ('idx_arm_temporal_term_a_delta',
     'CREATE INDEX IF NOT EXISTS idx_arm_temporal_term_a_delta '
     'ON public.fact_term_arm_temporal (term_a, delta_lift DESC NULLS LAST);'),
    ('idx_arm_temporal_term_b_delta',
     'CREATE INDEX IF NOT EXISTS idx_arm_temporal_term_b_delta '
     'ON public.fact_term_arm_temporal (term_b, delta_lift DESC NULLS LAST);'),
    ('idx_term_community_id',
     'CREATE INDEX IF NOT EXISTS idx_term_community_id '
     'ON public.dim_term_community (community_id);'),
]

with psycopg2.connect(DB_URL) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        print('=== CREATE INDEX (5) ===')
        for name, sql in INDEXES:
            t = time.time()
            cur.execute(sql)
            print(f'  ✓ {name}  ({time.time()-t:.1f}s)')

        print('\n=== ANALYZE ===')
        for table in PARQUETS:
            t = time.time()
            cur.execute(f'ANALYZE public.{table};')
            print(f'  ✓ ANALYZE {table}  ({time.time()-t:.1f}s)')

        print('\n=== Verify (row count parity) ===')
        all_ok = True
        for table in PARQUETS:
            cur.execute(f'SELECT COUNT(*) FROM public.{table}')
            n_db = cur.fetchone()[0]
            n_pq = pq.ParquetFile(PARQUETS[table]).metadata.num_rows
            ok = (n_db == n_pq)
            all_ok = all_ok and ok
            mark = '✅' if ok else '⚠'
            print(f'  {mark} {table}: db={n_db:,} parquet={n_pq:,} delta={n_db - n_pq:+,}')

        print('\n=== Sample: top-NPMI anchor 1-hop ===')
        cur.execute('''
            WITH anchor AS (
                SELECT term_a FROM public.fact_term_arm_static
                ORDER BY npmi DESC NULLS LAST LIMIT 1
            )
            SELECT term_a, term_b, npmi, lift
            FROM public.fact_term_arm_static
            WHERE term_a = (SELECT term_a FROM anchor)
               OR term_b = (SELECT term_a FROM anchor)
            ORDER BY npmi DESC NULLS LAST LIMIT 5;
        ''')
        for r in cur.fetchall():
            print(f'  {r[0]:<35} ↔ {r[1]:<35} NPMI={r[2]:.3f} lift={r[3]:.2f}')

        print('\n=== Sample: top trending (delta_lift DESC) ===')
        cur.execute('''
            SELECT term_a, term_b, delta_lift
            FROM public.fact_term_arm_temporal
            ORDER BY delta_lift DESC NULLS LAST LIMIT 5;
        ''')
        for r in cur.fetchall():
            print(f'  {r[0]:<35} ↔ {r[1]:<35} Δlift={r[2]:.3f}')

        print('\n=== Top-5 community by size ===')
        cur.execute('''
            SELECT community_id, COUNT(*) AS n_terms
            FROM public.dim_term_community
            GROUP BY community_id ORDER BY n_terms DESC LIMIT 5;
        ''')
        for r in cur.fetchall():
            print(f'  community_id={r[0]:<6} n_terms={r[1]}')

if all_ok:
    print('\n✅ Post-load tamam. 2.5 sayfası canlı bağlanmaya hazır.')
else:
    print('\n⚠ Row count parity FAIL — state.json + log incele.')